# Dataset

This notebook verifies that every structure selected in notebook 00 is on disk and can be read with the
chain recorded in `proteins.csv`, and then describes the dataset: which experimental methods each protein
has, the size of the entries its chains come from, the length of each chain, and how many structures were
public before the training cutoffs of AlphaFold2 and OpenFold3. These properties set the limits of what
the later comparisons can show.

**Input:** `proteins.csv` and `data/` from notebook 00; nothing is downloaded.  
**Output:** `results/dataset_structures.csv`.

In [1]:
import sys, os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from IPython.display import display

from src.parser import load_structure, get_ca_coords_and_seq

In [2]:
# Load the protein manifest
proteins = pd.read_csv("proteins.csv", dtype=str, keep_default_na=False)
proteins

,protein_name,uniprot_id,organism,mature_start,mature_end,pdb_xray,chain_xray,pdb_nmr,chain_nmr,pdb_cryoem,chain_cryoem,cov_xray,cov_nmr,cov_cryoem,res_xray,res_cryoem,notes
0,Type IV pilin PilE1,A0A0B7GU52,Streptococcus sanguinis,19,157,,,6I2O,A,8PFB,C,,0.813,0.942,,3.67,
1,Toxin co-regulated pilus biosynthesis protein ...,A0AAI7ZGD2,Xanthomonas citri pv. citri (strain 306),22,139,,,2L4W,A,6GYB,A,,0.983,0.941,,3.28,
2,ATP synthase epsilon chain (B1MLW3),B1MLW3,Mycobacteroides abscessus (strain ATCC 19977 /...,1,121,,,7VIL,A,9VKR,H,,0.992,1.0,,2.79,
3,Type II secretion system core protein G,E0SM38,Dickeya dadantii (strain 3937) (Erwinia chrysa...,8,153,,,29JD,A,11ME,B,,0.836,0.904,,3.6,
4,"Acyl carrier protein, mitochondrial",O14561,Homo sapiens (Human),69,156,6ODD,A,2DNW,A,7OF0,w,0.966,0.977,0.898,2.0,2.2,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,Dynein axonemal light chain 1,Q9XHH2,Chlamydomonas reinhardtii (Chlamydomonas smithii),1,198,5YXM,A,1DS9,A,8GLV,AA,0.98,1.0,0.985,1.545,3.1,
147,Cofilin-2 (Q9Y281),Q9Y281,Homo sapiens (Human),2,166,,,7M0G,A,9Y9P,a,,1.0,1.0,,2.06,
148,Ribosome maturation protein SBDS,Q9Y3A5,Homo sapiens (Human),2,250,,,2KDO,A,5AN9,J,,1.0,1.0,,3.3,
149,Peptidyl-prolyl cis-trans isomerase-like 1,Q9Y3C6,Homo sapiens (Human),1,166,2X7K,A,1XWN,A,9S2E,S,0.988,1.0,0.958,1.15,2.4,


## Verify all expected files are present

The table shows which CIF files are on disk. Any missing entry must be resolved by re-running notebook 00 before proceeding — the pipeline will silently skip missing files and record them in `results/missing_inputs.csv`.

In [3]:
rows = []

for _, row in proteins.iterrows():
    uid = row["uniprot_id"]
    checks = [
        (
            "X-ray",
            f"data/experimental/{str(row.get('pdb_xray',   '') or '').upper()}.cif",
        ),
        (
            "NMR",
            f"data/experimental/{str(row.get('pdb_nmr',    '') or '').upper()}.cif",
        ),
        (
            "Cryo-EM",
            f"data/experimental/{str(row.get('pdb_cryoem', '') or '').upper()}.cif",
        ),
        ("AF2", f"data/alphafold2/AF2_{uid}.cif"),
        ("OF3", f"data/openfold3/OF3_{uid}.cif"),
    ]
    for method, path in checks:
        fname = os.path.basename(path)
        if fname.split(".")[0].upper() in ("", "NAN", "NONE"):
            continue  # method not available for this protein
        rows.append(
            {
                "protein": row["protein_name"],
                "method": method,
                "file": fname,
                "status": "OK" if os.path.exists(path) else "MISSING",
            }
        )

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print()
n_missing = (summary["status"] == "MISSING").sum()
if n_missing == 0:
    print("All expected files present.")
else:
    print(f"{n_missing} file(s) missing — run 00_data_acquisition.ipynb first.")

                                                                             protein  method               file status
                                                                 Type IV pilin PilE1     NMR           6I2O.cif     OK
                                                                 Type IV pilin PilE1 Cryo-EM           8PFB.cif     OK
                                                                 Type IV pilin PilE1     AF2 AF2_A0A0B7GU52.cif     OK
                                                                 Type IV pilin PilE1     OF3 OF3_A0A0B7GU52.cif     OK
Toxin co-regulated pilus biosynthesis protein Q C-terminal domain-containing protein     NMR           2L4W.cif     OK
Toxin co-regulated pilus biosynthesis protein Q C-terminal domain-containing protein Cryo-EM           6GYB.cif     OK
Toxin co-regulated pilus biosynthesis protein Q C-terminal domain-containing protein     AF2 AF2_A0AAI7ZGD2.cif     OK
Toxin co-regulated pilus biosynthesis protein Q 

## Sanity check — inspect one structure per type

For each protein, load every structure with the chain chosen for it in `proteins.csv`
(one chain per method; predictions are chain A) and print: number of models,
number of chains in the entry, Cα count of the chosen chain, and its first 20 residues.


In [4]:
CHAIN_COLUMN = {"X-ray": "chain_xray", "NMR": "chain_nmr", "Cryo-EM": "chain_cryoem"}
PDB_COLUMN = {"X-ray": "pdb_xray", "NMR": "pdb_nmr", "Cryo-EM": "pdb_cryoem"}


def structures_of(row):
    """(label, path, model index, chain) for every structure of one manifest row."""
    uid = row["uniprot_id"]
    items = []
    for label, col in PDB_COLUMN.items():
        pdb_id = row[col].strip()
        if pdb_id:
            items.append(
                (
                    label,
                    f"data/experimental/{pdb_id.upper()}.cif",
                    0,
                    row[CHAIN_COLUMN[label]],
                )
            )
    items.append(("AF2", f"data/alphafold2/AF2_{uid}.cif", 0, "A"))
    items.append(("OF3", f"data/openfold3/OF3_{uid}.cif", 0, "A"))
    return items


for _, row in proteins.iterrows():
    print(f"\n{'='*55}")
    print(f"  {row['protein_name']}  ({row['uniprot_id']})")
    print(f"{'='*55}")

    for label, path, model_idx, chain_id in structures_of(row):
        if not os.path.exists(path):
            print(f"  {label:<8}  [MISSING] {path}")
            continue
        try:
            struct = load_structure(path)
            coords, seq = get_ca_coords_and_seq(
                struct, model_idx=model_idx, chain_id=chain_id
            )
            print(
                f"  {label:<8}  chain={chain_id}  models={len(struct)}  "
                f"chains in entry={len(struct[0])}  Ca={len(coords)}  seq={seq[:20]}..."
            )
        except Exception as exc:
            print(f"  {label:<8}  [ERROR] chain {chain_id}: {exc}")


  Type IV pilin PilE1  (A0A0B7GU52)
  NMR       chain=A  models=10  chains in entry=1  Ca=113  seq=HQDNARKSRIQSEHRELVSA...
  Cryo-EM   chain=C  models=1  chains in entry=9  Ca=131  seq=FTLVELIVVIIIIAIIAAVA...
  AF2       chain=A  models=1  chains in entry=1  Ca=157  seq=MLNKLQKFRQDLKKKGKGFT...
  OF3       chain=A  models=1  chains in entry=1  Ca=157  seq=MLNKLQKFRQDLKKKGKGFT...

  Toxin co-regulated pilus biosynthesis protein Q C-terminal domain-containing protein  (A0AAI7ZGD2)
  NMR       chain=A  models=20  chains in entry=1  Ca=120  seq=GSHMTKPAPDFGGRWKHVNH...
  Cryo-EM   chain=A  models=1  chains in entry=42  Ca=111  seq=CATKPAPDFGGRWKHVNHFD...
  AF2       chain=A  models=1  chains in entry=1  Ca=139  seq=MNPMYVSKLSLVLVAAALVG...
  OF3       chain=A  models=1  chains in entry=1  Ca=139  seq=MNPMYVSKLSLVLVAAALVG...

  ATP synthase epsilon chain (B1MLW3)  (B1MLW3)
  NMR       chain=A  models=20  chains in entry=1  Ca=120  seq=SEIDVEIVAVEREIWSGKAT...
  Cryo-EM   chain=H  models=1  cha

## Sequence length comparison

Number of Cα residues of each structure next to the mature chain length from UniProt.
AlphaFold DB models cover the full precursor (signal peptide, propeptide) while experimental
chains contain the mature protein, so AF2 can be longer than the mature column.
Coverage against the mature chain was already checked when the dataset was built (notebook 00).


In [5]:
length_rows = []

for _, row in proteins.iterrows():
    name = row["protein_name"]
    mature = int(row["mature_end"]) - int(row["mature_start"]) + 1
    length_rows.append({"protein": name, "source": "mature (UniProt)", "n_ca": mature})

    for label, path, model_idx, chain_id in structures_of(row):
        if not os.path.exists(path):
            length_rows.append({"protein": name, "source": label, "n_ca": None})
            continue
        try:
            struct = load_structure(path)
            coords, _ = get_ca_coords_and_seq(
                struct, model_idx=model_idx, chain_id=chain_id
            )
            length_rows.append({"protein": name, "source": label, "n_ca": len(coords)})
        except Exception as exc:
            print(f"  [ERROR] {name} {label} chain {chain_id}: {exc}")
            length_rows.append({"protein": name, "source": label, "n_ca": None})

pd.DataFrame(length_rows).pivot(index="protein", columns="source", values="n_ca")

source,AF2,Cryo-EM,NMR,OF3,X-ray,mature (UniProt)
protein,,,,,,
ADP-ribosylation factor 1 (P11076),181.0,163.0,180.0,181.0,NaN,180.0
ADP-ribosylation factor 1 (P84077),181.0,165.0,164.0,181.0,170.0,180.0
ATP synthase epsilon chain (B1MLW3),121.0,121.0,120.0,121.0,NaN,121.0
ATP synthase epsilon chain (P0A6E6),139.0,136.0,138.0,139.0,128.0,138.0
ATP synthase epsilon chain (P9WPV1),121.0,118.0,121.0,121.0,NaN,120.0
...,...,...,...,...,...,...
Ubiquitin-conjugating enzyme E2 L3,154.0,153.0,154.0,154.0,154.0,154.0
Ubiquitin-fold modifier 1,85.0,78.0,88.0,85.0,82.0,83.0
Ubiquitin-like protein 5,73.0,73.0,73.0,73.0,76.0,73.0


## Cryo-EM method check

The wwPDB method table used in notebook 00 files electron crystallography (MicroED) under "EM" too.
MicroED is a diffraction method, not single-particle cryo-EM: if an entry below is listed,
choose the next-best cryo-EM chain for that protein.

In [6]:
from src.metadata import extract_metadata

not_cryo_em = []
for _, row in proteins[proteins["pdb_cryoem"] != ""].iterrows():
    path = f"data/experimental/{row['pdb_cryoem'].upper()}.cif"
    if not os.path.exists(path):
        continue
    meta = extract_metadata(
        path,
        uniprot_id=row["uniprot_id"],
        pdb_id=row["pdb_cryoem"],
        chain_id=row["chain_cryoem"],
    )
    if meta.method != "ELECTRON MICROSCOPY":
        not_cryo_em.append(
            {
                "protein": row["protein_name"],
                "pdb_cryoem": row["pdb_cryoem"],
                "method": meta.method,
            }
        )

print(
    pd.DataFrame(not_cryo_em)
    if not_cryo_em
    else "All cryo-EM entries are ELECTRON MICROSCOPY."
)

All cryo-EM entries are ELECTRON MICROSCOPY.


---
## What the set is — and what it can test

Three properties of the set shape every later result:

1. **How many experimental pairs a protein has.** The set is built on NMR ∩ cryo-EM; X-ray is
   present only for part of it. A protein's experimental variability is therefore either one pair
   (NMR–cryo-EM) or three (X-ray–NMR, X-ray–cryo-EM, NMR–cryo-EM).
2. **Context.** A cryo-EM chain is often one subunit of a large assembly, while NMR measures the
   protein alone. Part of a difference between methods can then be a difference between "in a
   complex" and "alone" (notebook 04 splits the results by assembly size).
3. **Training overlap.** AlphaFold2 was trained on PDB entries released up to 30 April 2018,
   OpenFold3 on entries deposited before 30 September 2021. A protein with a structure public before
   a model's cutoff may have been in its training data, and agreement with that structure can partly
   be recall. The set then measures agreement on proteins the models may have seen, not prediction
   of new ones.

In [7]:
import src.dataset as ds
from src.analysis import AF2_TRAINING_CUTOFF, OF3_TRAINING_CUTOFF, training_overlap

os.makedirs("results", exist_ok=True)
print("Experimental methods per protein:")
display(ds.composition(proteins))

described = ds.describe_structures(proteins, "data")
described.to_csv("results/dataset_structures.csv", index=False)
ok = described[described["status"] == "ok"].copy()
print(
    f"{len(ok)} of {len(described)} experimental structures read (results/dataset_structures.csv)"
)
display(
    ok.groupby("method").agg(
        structures=("pdb_id", "size"),
        median_residues=("n_residues", "median"),
        median_resolution_A=("resolution", "median"),
        entries_with_several_models=("n_models", lambda s: int((s > 1).sum())),
        median_polymer_chains=("n_polymer_chains", "median"),
    )
)

Experimental methods per protein:


,experimental methods,proteins
0,X-ray + NMR + Cryo-EM,90
1,NMR + Cryo-EM,61


392 of 392 experimental structures read (results/dataset_structures.csv)


,structures,median_residues,median_resolution_A,entries_with_several_models,median_polymer_chains
method,,,,,
Cryo-EM,151,116.0,2.880,0,10.0
NMR,151,121.0,NaN,131,1.0
X-ray,90,122.5,1.645,0,2.0


In [8]:
# 2. Context: in how large an entry was each chain determined?
bins = [0, 1, 4, 19, np.inf]
labels = ["1 (alone)", "2–4", "5–19", "20+"]
context = ok.assign(
    chains=pd.cut(pd.to_numeric(ok["n_polymer_chains"]), bins, labels=labels)
).pivot_table(
    index="chains",
    columns="method",
    values="pdb_id",
    aggfunc="count",
    fill_value=0,
    observed=False,
)
display(context)
em = pd.to_numeric(ok.loc[ok["method"] == "Cryo-EM", "n_polymer_chains"])
nmr = pd.to_numeric(ok.loc[ok["method"] == "NMR", "n_polymer_chains"])
print(
    f"Cryo-EM: {int((em > 1).sum())} of {len(em)} chains come from entries with more than one polymer "
    f"chain (median {em.median():.0f}); NMR: the protein alone in {int((nmr == 1).sum())} of {len(nmr)} entries."
)

# 3. Training overlap: release date against the AF2 cutoff, deposition date against the OF3 cutoff
PERIODS = [
    "before both cutoffs",
    "before the OF3 cutoff only",
    "after both cutoffs",
    "date unknown",
]
period = np.select(
    [
        ok["release_date"].eq("") | ok["deposition_date"].eq(""),
        ok["before_af2_cutoff"],
        ok["before_of3_cutoff"],
    ],
    ["date unknown", "before both cutoffs", "before the OF3 cutoff only"],
    default="after both cutoffs",
)
print(
    f"AF2 cutoff: released ≤ {AF2_TRAINING_CUTOFF}; OF3 cutoff: deposited < {OF3_TRAINING_CUTOFF}"
)
entry = pd.Series(
    pd.Categorical(period, categories=PERIODS), index=ok.index, name="entry"
)
display(pd.crosstab(entry, ok["method"], dropna=False))
overlap = training_overlap(ok)
n = len(overlap)
unseen_af2 = int((~overlap["seen_af2"] & overlap["dated"]).sum())
unseen_of3 = int((~overlap["seen_of3"] & overlap["dated"]).sum())
print(
    f"Proteins with at least one structure here that was public before the cutoff: "
    f"AF2 {int(overlap['seen_af2'].sum())} of {n}, OF3 {int(overlap['seen_of3'].sum())} of {n}."
)
print(
    f"Proteins whose structures here all came after the cutoff: AF2 {unseen_af2}, OF3 {unseen_of3} "
    "(upper bounds: older PDB entries of the same protein outside this set are not counted)."
)

method,Cryo-EM,NMR,X-ray
chains,,,
1 (alone),2,116,28
2–4,29,32,48
5–19,72,3,11
20+,48,0,3


Cryo-EM: 149 of 151 chains come from entries with more than one polymer chain (median 10); NMR: the protein alone in 116 of 151 entries.
AF2 cutoff: released ≤ 2018-04-30; OF3 cutoff: deposited < 2021-09-30


method,Cryo-EM,NMR,X-ray
entry,,,
before both cutoffs,6,132,62
before the OF3 cutoff only,32,15,9
after both cutoffs,113,4,19
date unknown,0,0,0


Proteins with at least one structure here that was public before the cutoff: AF2 137 of 151, OF3 149 of 151.
Proteins whose structures here all came after the cutoff: AF2 14, OF3 2 (upper bounds: older PDB entries of the same protein outside this set are not counted).


**Reading.** Keep the three numbers above in mind for every result in notebooks 03–05: how many
proteins have only one experimental pair, how many cryo-EM chains come from assemblies, and how few
proteins the models cannot have seen. The last one decides what the thesis can claim: agreement
between predictions and experiments on proteins that were (mostly) available to the models during
training is evidence that the models reproduce known structures, not that they would replace an
experiment on a new protein.